# Stage 04: Lesion Segmentation

**Status:** `lesion_segmentation_dataset.py` / `lesion_segmentation_model.py` are implemented and
locally smoke-tested against real IDRiD images/masks and the real Stage 03 checkpoint (see
`SEGMENTATION_ARCHITECTURE.md` Sec 3 for the frozen design). This notebook prepares that
already-verified implementation for its first real Colab training run -- it does not change
`lesion_segmentation_dataset.py` / `lesion_segmentation_model.py` / `config.py`, and it does not
start training automatically.

## Objective

Train an Attention U-Net (`lesion_segmentation_model.build_attention_unet`) on IDRiD's
segmentation subset to segment four diabetic retinopathy lesion classes -- Microaneurysm,
Haemorrhage, Hard Exudate, Soft Exudate -- from a 4-channel input (Stage 02 processed RGB +
Stage 03 vessel probability map). Unlike Stage 03, this stage has a full training workflow of
its own within this project.

## Expected Inputs

- Stage 02 processed RGB images for IDRiD's segmentation subset (`datasets/IDRiD/segmentation/processed/`,
  already generated by `stage02_preprocessing.ipynb` -- this notebook does **not** rerun Stage 02).
- IDRiD's real lesion ground-truth masks (`datasets/IDRiD/segmentation/raw/2. All Segmentation
  Groundtruths/`) -- only the Microaneurysm/Haemorrhage/Hard Exudate/Soft Exudate categories;
  Optic Disc is never staged or used as a target (`SEGMENTATION_ARCHITECTURE.md` Sec 3.4).
- The vendored Stage 03 LWNet checkpoint, already uploaded to
  `exported_models/VesselSegmentation/{best_model.pth,config.cfg}` on Drive (Section 4 below
  stages a local copy; it does not re-derive or retrain it).

## Expected Outputs

- A trained Attention U-Net exported to `exported_models/LesionSegmentation/best_model.keras` on
  Drive, plus a full training run archived under `experiments/LesionSegmentation/<timestamp>/`
  (checkpoints, logs, evaluation).
- Real per-class + mean Dice/IoU on the official, held-out 27-image IDRiD Testing Set --
  never fabricated, never computed until a real trained model exists.

## Datasets

IDRiD's segmentation subset only (54-image Training Set, further split 80/20 train/validation;
27-image Testing Set, held out entirely). No other dataset is staged by this notebook.

## Dependencies

Stage 02 (already run for IDRiD/segmentation -- see `stage02_preprocessing.ipynb`'s own logged
run). Stage 03's vendored checkpoint (already integrated -- see `stage03_vessel_segmentation.ipynb`).

## Workflow

1. Bootstrap
2. Setup
3. Environment verification
4. Stage 03 vessel checkpoint staging + verification
5. Stage 02 processed IDRiD/segmentation image staging
6. IDRiD raw lesion ground-truth (+ filename) staging
7. Dataset discovery and verification
8. Dataset creation
9. Model creation
10. Training setup
11. Training (**prepared, not auto-started** -- see Section 11's `RUN_TRAINING` flag)
12. Evaluation (only after training, on the held-out 27-image test set)
13. Export
14. Final summary

## Before running

`Runtime > Change runtime type > Hardware accelerator > GPU` -- unlike Stage 03, this stage
trains for real and requires a GPU. Confirm your Drive contains
`MyDrive/DiabeticRetinopathy/exported_models/VesselSegmentation/{best_model.pth,config.cfg}` and
that Stage 02 has already been run for `datasets/IDRiD/segmentation` (see
`stage02_preprocessing.ipynb`).

**Storage rule:** the full dataset is never copied into this git repository. Google Drive is the
only persistent dataset/output store; the Colab local SSD (`/content/...`) is temporary staging
only, discarded when the session ends.

### 1. Bootstrap

Same minimal clone + `sys.path` setup every stage notebook needs -- see `colab/common/setup.py`'s
module docstring for why this is intentionally duplicated.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

## 2. Setup

`setup.setup()` mounts Google Drive, clones/updates the repository, installs `requirements.txt`,
and enters the repository -- identical call to every other stage notebook, reused unmodified.

In [ ]:
import setup

setup_info = setup.setup()

## 3. Environment Verification

`require_gpu=True` -- unlike Stage 03 (a small pretrained inference-only model), Stage 04 trains
an Attention U-Net for real and needs a GPU runtime.

In [ ]:
import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=True,
)

## 4. Stage 03 Vessel Checkpoint Staging + Verification

Copies **only** the two vendored LWNet artifacts (`best_model.pth`, `config.cfg`) from Drive's
`exported_models/VesselSegmentation/` to this project's own `models/vessel_segmentation/` on the
Colab VM's **local SSD** -- identical to `stage03_vessel_segmentation.ipynb`'s own Section 4 (not
duplicated logic, the same small pattern repeated because this stage's dataset loader needs the
checkpoint at that same local path). `vessel_segmentation_model.py` / `vessel_segmentation_inference.py`
are not modified here -- this cell only loads the already-implemented, already-verified
`load_vessel_model()` and checks the result (parameter count, `mode`/`training` flags, device), the
same verification `stage03_vessel_segmentation.ipynb` Section 5 performs.

**One-time manual prerequisite:** `best_model.pth` + `config.cfg` must already be uploaded to
`MyDrive/DiabeticRetinopathy/exported_models/VesselSegmentation/` (see
`stage03_vessel_segmentation.ipynb`'s own Section 4 for the full one-time upload note).

In [ ]:
import shutil

import config
from vessel_segmentation_inference import DEFAULT_MODEL_PATH as DEFAULT_VESSEL_MODEL_PATH
from vessel_segmentation_inference import load_vessel_model

VESSEL_SEG_DRIVE_DIR = colab_config.DRIVE.exported_model_dir("VesselSegmentation")
VESSEL_CHECKPOINT_FILES = ("best_model.pth", "config.cfg")

os.makedirs(config.VESSEL_SEG_MODEL_DIR, exist_ok=True)
for filename in VESSEL_CHECKPOINT_FILES:
    src = os.path.join(VESSEL_SEG_DRIVE_DIR, filename)
    dst = os.path.join(config.VESSEL_SEG_MODEL_DIR, filename)
    if not os.path.isfile(src):
        raise RuntimeError(
            f"Vessel Segmentation artifact not found on Drive: {src}. Upload the vendored "
            "LWNet checkpoint there first -- see this cell's markdown and "
            "SEGMENTATION_ARCHITECTURE.md Sec 6."
        )
    shutil.copy2(src, dst)
    print(f"Staged {src} -> {dst} ({os.path.getsize(dst):,} bytes)")

vessel_model = load_vessel_model(DEFAULT_VESSEL_MODEL_PATH)
n_vessel_params = sum(p.numel() for p in vessel_model.parameters())
vessel_model_device = next(vessel_model.parameters()).device
print(f"\nLoaded vessel checkpoint from {DEFAULT_VESSEL_MODEL_PATH}")
print(f"Parameter count: {n_vessel_params:,} (LWNet's own paper reports ~70k)")
print(f"Model device: {vessel_model_device}")
print(f"model.mode = {vessel_model.mode!r} (must be 'eval')")
print(f"model.training = {vessel_model.training} (must be False)")
assert vessel_model.mode == "eval"
assert vessel_model.training is False
print("Verified: Stage 03 vessel checkpoint loads correctly and is ready for Stage 04 to use.")

## 5. Stage 02 Processed IDRiD/Segmentation Image Staging

Stages the **already-generated** Stage 02 output for IDRiD's segmentation subset -- both the
54-image Training Set and 27-image Testing Set under `1. Original Images/` -- from Drive to the
Colab VM's local SSD via `dataset_staging.stage_dataset()` (reused unmodified; see
`colab/common/dataset_staging.py`). This notebook **never calls `image_preprocessing.py`** and
never regenerates this output -- per `PROJECT_CODE.md`'s Stage 02 Preprocessing Policy ("no
downstream stage should regenerate deterministic preprocessing outputs").

In [ ]:
import posixpath

import dataset_staging
import lesion_segmentation_dataset as lsd

IDRID_SEG_RAW_DRIVE_DIR = posixpath.join(colab_config.IDRID_DATASET_DIR, "segmentation", "raw")
IDRID_SEG_PROCESSED_DRIVE_DIR = posixpath.join(colab_config.IDRID_DATASET_DIR, "segmentation", "processed")

RAW_LOCAL_ROOT = posixpath.join(dataset_staging.LOCAL_DATASETS_ROOT, "IDRiD_segmentation_raw")
PROCESSED_LOCAL_ROOT = posixpath.join(dataset_staging.LOCAL_DATASETS_ROOT, "IDRiD_segmentation_processed")

for split_dir in (lsd.TRAIN_SPLIT_DIR, lsd.TEST_SPLIT_DIR):
    staged = dataset_staging.stage_dataset(
        posixpath.join(IDRID_SEG_PROCESSED_DRIVE_DIR, lsd._ORIGINAL_IMAGES_SUBDIR, split_dir),
        f"IDRiD_segmentation_processed/{lsd._ORIGINAL_IMAGES_SUBDIR}/{split_dir}",
        local_root=dataset_staging.LOCAL_DATASETS_ROOT,
    )
    dataset_staging.verify_staged_copy(staged)

print(f"\nStage 02 processed images staged under: {PROCESSED_LOCAL_ROOT}")

## 6. IDRiD Raw Lesion Ground-Truth (+ Filename) Staging

Two things are staged here, both from `datasets/IDRiD/segmentation/raw/` (never written to):

1. **The raw `1. Original Images/` JPEGs** (both splits) -- needed only so
   `lesion_segmentation_dataset._list_original_images()` can discover which image ids/filenames
   exist (the actual pixel content used for training comes from Section 5's Stage 02 output, not
   from these raw copies).
2. **Exactly four lesion mask categories** -- Microaneurysm, Haemorrhage, Hard Exudate, Soft
   Exudate -- for both splits. **Optic Disc (`5. Optic Disc`) is never staged**, matching
   `SEGMENTATION_ARCHITECTURE.md` Sec 3.4/Appendix A.4 (it is not one of this stage's output
   classes). `lesion_segmentation_dataset._MASK_SPECS` (imported, not restated) is the single
   source of truth for which four categories these are.

Every call reuses `dataset_staging.stage_dataset()`/`verify_staged_copy()` unmodified -- staging
each subfolder under a `dataset_name` that encodes its own relative path (e.g. `"IDRiD_segmentation_raw/
2. All Segmentation Groundtruths/a. Training Set/1. Microaneurysms"`) reproduces exactly the
directory shape `lesion_segmentation_dataset.py` expects under a single local `raw_dir` root,
without needing any new staging logic.

In [ ]:
for split_dir in (lsd.TRAIN_SPLIT_DIR, lsd.TEST_SPLIT_DIR):
    staged = dataset_staging.stage_dataset(
        posixpath.join(IDRID_SEG_RAW_DRIVE_DIR, lsd._ORIGINAL_IMAGES_SUBDIR, split_dir),
        f"IDRiD_segmentation_raw/{lsd._ORIGINAL_IMAGES_SUBDIR}/{split_dir}",
        local_root=dataset_staging.LOCAL_DATASETS_ROOT,
    )
    dataset_staging.verify_staged_copy(staged)

for split_dir in (lsd.TRAIN_SPLIT_DIR, lsd.TEST_SPLIT_DIR):
    for class_name, category_folder, suffix in lsd._MASK_SPECS:
        staged = dataset_staging.stage_dataset(
            posixpath.join(IDRID_SEG_RAW_DRIVE_DIR, lsd._GROUNDTRUTHS_SUBDIR, split_dir, category_folder),
            f"IDRiD_segmentation_raw/{lsd._GROUNDTRUTHS_SUBDIR}/{split_dir}/{category_folder}",
            local_root=dataset_staging.LOCAL_DATASETS_ROOT,
        )
        dataset_staging.verify_staged_copy(staged)

print(f"\nStaged categories (Optic Disc deliberately excluded): {[c[0] for c in lsd._MASK_SPECS]}")
print(f"Raw images + lesion masks staged under: {RAW_LOCAL_ROOT}")

## 7. Dataset Discovery and Verification

Configuration constants below are read directly from `lesion_segmentation_dataset.py`'s own
approved defaults (`DEFAULT_IMAGE_SIZE`, `DEFAULT_VAL_SPLIT`, `DEFAULT_SEED`, `DEFAULT_VESSEL_CACHE_DIR`)
rather than restated here, so this notebook can never silently drift from the frozen Steps 0-3
implementation. The checks below run the real, unmodified dataset-loader functions against the
now-staged local data -- no dataset name, count, or matching rule is hardcoded here beyond what
`lesion_segmentation_dataset.py` itself already encodes.

In [ ]:
IMAGE_SIZE = lsd.DEFAULT_IMAGE_SIZE          # frozen default (512, 512) -- not changed here
VAL_SPLIT = lsd.DEFAULT_VAL_SPLIT            # frozen default 0.2 -- not changed here
RANDOM_SEED = lsd.DEFAULT_SEED               # frozen default 42 -- not changed here
VESSEL_CACHE_DIR = lsd.DEFAULT_VESSEL_CACHE_DIR  # frozen cache location -- not changed here

print(f"IMAGE_SIZE={IMAGE_SIZE}  VAL_SPLIT={VAL_SPLIT}  RANDOM_SEED={RANDOM_SEED}")
print(f"VESSEL_CACHE_DIR={VESSEL_CACHE_DIR}")

In [ ]:
train_entries = lsd._list_original_images(RAW_LOCAL_ROOT, lsd.TRAIN_SPLIT_DIR)
test_entries = lsd._list_original_images(RAW_LOCAL_ROOT, lsd.TEST_SPLIT_DIR)

print(f"Training images discovered: {len(train_entries)} (official IDRiD segmentation Training Set)")
print(f"Testing images discovered:  {len(test_entries)} (official IDRiD segmentation Testing Set)")

assert len(train_entries) == 54, f"expected the official 54-image Training Set, found {len(train_entries)}"
assert len(test_entries) == 27, f"expected the official 27-image Testing Set, found {len(test_entries)}"
print("[PASS] 54 training images and 27 testing images verified.")

In [ ]:
def _mask_coverage(split_dir, entries):
    coverage = {}
    for class_name, category_folder, suffix in lsd._MASK_SPECS:
        present = sum(
            1 for image_id, _ in entries
            if os.path.exists(lsd._mask_path(RAW_LOCAL_ROOT, split_dir, category_folder, suffix, image_id))
        )
        coverage[class_name] = present
    return coverage

train_coverage = _mask_coverage(lsd.TRAIN_SPLIT_DIR, train_entries)
test_coverage = _mask_coverage(lsd.TEST_SPLIT_DIR, test_entries)

print("Mask coverage (Training Set, out of 54):")
for name, count in train_coverage.items():
    print(f"  {name}: {count}")
print("Mask coverage (Testing Set, out of 27):")
for name, count in test_coverage.items():
    print(f"  {name}: {count}")

for name in lsd.LESION_CLASSES:
    assert train_coverage[name] > 0, f"expected at least one real {name} mask in the Training Set"
    assert test_coverage[name] > 0, f"expected at least one real {name} mask in the Testing Set"
print("\n[PASS] every lesion class has at least some real ground truth in both splits "
      "(Soft Exudate is expected to be the sparsest -- see Section 14's caveats).")

In [ ]:
soft_exudate_spec = next(spec for spec in lsd._MASK_SPECS if spec[0] == "SoftExudate")
_, se_folder, se_suffix = soft_exudate_spec

missing_se_id, present_se_id = None, None
for image_id, _ in train_entries:
    has_se = os.path.exists(lsd._mask_path(RAW_LOCAL_ROOT, lsd.TRAIN_SPLIT_DIR, se_folder, se_suffix, image_id))
    if has_se and present_se_id is None:
        present_se_id = image_id
    if not has_se and missing_se_id is None:
        missing_se_id = image_id
    if missing_se_id and present_se_id:
        break

train_filenames = dict(train_entries)

if missing_se_id is not None:
    x_missing, y_missing = lsd._build_sample(
        missing_se_id, train_filenames[missing_se_id], RAW_LOCAL_ROOT, PROCESSED_LOCAL_ROOT,
        lsd.TRAIN_SPLIT_DIR, VESSEL_CACHE_DIR, vessel_model, IMAGE_SIZE,
    )
    se_channel = y_missing[..., lsd.LESION_CLASSES.index("SoftExudate")]
    assert se_channel.sum() == 0.0, f"expected an all-zero SoftExudate channel for IDRiD_{missing_se_id}"
    print(f"[PASS] IDRiD_{missing_se_id} has no real Soft Exudate mask file -> "
          "SoftExudate channel is all-zero, as expected.")
else:
    print("[INFO] every discovered training image has a Soft Exudate mask -- nothing to verify here.")

if present_se_id is not None:
    x_present, y_present = lsd._build_sample(
        present_se_id, train_filenames[present_se_id], RAW_LOCAL_ROOT, PROCESSED_LOCAL_ROOT,
        lsd.TRAIN_SPLIT_DIR, VESSEL_CACHE_DIR, vessel_model, IMAGE_SIZE,
    )
    se_channel = y_present[..., lsd.LESION_CLASSES.index("SoftExudate")]
    assert se_channel.sum() > 0.0, f"expected a nonzero SoftExudate channel for IDRiD_{present_se_id}"
    print(f"[PASS] IDRiD_{present_se_id} has a real Soft Exudate mask -> SoftExudate channel is nonzero.")

In [ ]:
split_train_ids, split_val_ids = lsd.split_train_val_ids(RAW_LOCAL_ROOT, val_split=VAL_SPLIT, seed=RANDOM_SEED)
test_ids = {image_id for image_id, _ in test_entries}

print(f"Train ids: {len(split_train_ids)}  Val ids: {len(split_val_ids)}  Test ids: {len(test_ids)}")

assert set(split_train_ids) & set(split_val_ids) == set(), "train/val ids must not overlap"
assert set(split_train_ids) & test_ids == set(), "train ids must not overlap the official test set"
assert set(split_val_ids) & test_ids == set(), "val ids must not overlap the official test set"
print("[PASS] train/validation/test separation verified -- no id appears in more than one split.")

## 8. Dataset Creation

Builds the real train/validation/test `tf.data.Dataset` pipelines via
`lesion_segmentation_dataset.load_lesion_segmentation_datasets()` /
`load_lesion_segmentation_test_dataset()` (unmodified), reusing the already-loaded `vessel_model`
from Section 4 (so Stage 03 inference is never re-triggered by a redundant checkpoint load).
Pulling one batch below (`next(iter(train_ds))`) verifies the 4-channel input/target contract on
real, staged data -- it processes only that one batch's worth of images, not the full dataset.

In [ ]:
BATCH_SIZE = 4

train_ds, val_ds = lsd.load_lesion_segmentation_datasets(
    raw_dir=RAW_LOCAL_ROOT, processed_dir=PROCESSED_LOCAL_ROOT,
    vessel_model=vessel_model, image_size=IMAGE_SIZE, val_split=VAL_SPLIT,
    batch_size=BATCH_SIZE, seed=RANDOM_SEED,
)
test_ds = lsd.load_lesion_segmentation_test_dataset(
    raw_dir=RAW_LOCAL_ROOT, processed_dir=PROCESSED_LOCAL_ROOT,
    vessel_model=vessel_model, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
)

x_batch, y_batch = next(iter(train_ds))
print(f"Train input batch shape:  {x_batch.shape}")
print(f"Train target batch shape: {y_batch.shape}")
assert x_batch.shape[-1] == 4, "expected a 4-channel input (RGB + vessel probability map)"
assert y_batch.shape[-1] == 4, "expected a 4-channel target (Microaneurysm/Haemorrhage/HardExudate/SoftExudate)"
print("[PASS] dataset creation verified: 4-channel input, 4-channel target.")

## 9. Model Creation

Builds the Attention U-Net once here to inspect the architecture and verify its I/O contract
before committing to a full run -- mirrors `stage01_iqa.ipynb`'s own "preview model, then
discard" pattern (Section 4 there). Section 11 builds its own instance via
`LesionSegmentationStage`, not this preview.

In [ ]:
from lesion_segmentation_model import LESION_CLASSES, build_attention_unet

preview_model = build_attention_unet(input_shape=(*IMAGE_SIZE, 4))
preview_model.summary()

preview_output = preview_model.predict(x_batch, verbose=0)
print(f"\nPrediction shape: {preview_output.shape}")
print(f"Lesion classes (output-channel order): {list(LESION_CLASSES)}")
assert preview_output.shape == (x_batch.shape[0], *IMAGE_SIZE, 4)
assert preview_output.min() >= 0.0 and preview_output.max() <= 1.0
print("[PASS] model I/O verified: (H, W, 4) sigmoid output, values in [0, 1].")

del preview_model, preview_output

## 10. Training Setup

Re-confirms Section 3's GPU check before proceeding (this stage trains for real), then creates a
new, isolated, timestamped experiment folder under `experiments/LesionSegmentation/` via
`experiment_manager.resolve_experiment()` (identical mechanism to every other stage notebook --
never overwrites a previous run). `EPOCHS`/`LEARNING_RATE` are this notebook's own training
hyperparameters (not part of the frozen Steps 0-3 defaults, which only cover image size/val
split/threshold/vessel cache/resize policy) -- adjust them here if needed before Section 11.

In [ ]:
import experiment_manager
from lesion_segmentation_model import LesionSegmentationStage

assert env_report["gpu_count"] > 0, "Stage 04 training requires a GPU runtime -- see Section 3."

EPOCHS = 100
LEARNING_RATE = 1e-4
RESUME_EXPERIMENT_DIR = None  # set to an existing experiment's root path to resume that run

experiment = experiment_manager.resolve_experiment(
    colab_config.DRIVE.experiment_dir("LesionSegmentation"),
    colab_config.REPO_DIR,
    resume_from=RESUME_EXPERIMENT_DIR,
    dataset_path=PROCESSED_LOCAL_ROOT,
    image_size=list(IMAGE_SIZE),
    val_split=VAL_SPLIT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    random_seed=RANDOM_SEED,
)

stage = LesionSegmentationStage(input_shape=(*IMAGE_SIZE, 4))
print(f"Experiment root: {experiment.root}")

## 11. Training

**Prepared, not auto-started.** `RUN_TRAINING` defaults to `False` -- running this notebook top
to bottom (including this cell) does **not** start training. Set `RUN_TRAINING = True` below and
re-run this cell only when you are ready to begin the real training run. Uses the current
approved defaults from Sections 7-10 (`IMAGE_SIZE`, `VAL_SPLIT`, `EPOCHS`, `LEARNING_RATE`,
`BATCH_SIZE`) and `LesionSegmentationStage.train()` (which itself uses `training.Trainer`/
`TrainingConfig` -- checkpointing, early stopping, `ReduceLROnPlateau`, TensorBoard, resume
support, all unmodified).

In [ ]:
RUN_TRAINING = False  # Set to True and re-run this cell when ready to start real training.

if not RUN_TRAINING:
    print("RUN_TRAINING is False -- training is prepared but NOT started.")
    print("Set RUN_TRAINING = True above and re-run this cell when you are ready.")
    history = None
else:
    history = stage.train(
        train_ds, val_ds,
        run_dir=experiment.root,
        epochs=EPOCHS,
        resume=RESUME_EXPERIMENT_DIR is not None,
    )
    experiment_manager.archive_tensorboard_logs(experiment)
    print("\nTraining complete.")

## 12. Evaluation

Runs **only after training** (`history is not None`), against the official, held-out 27-image
IDRiD segmentation Testing Set (`test_ds` from Section 8 -- never used for training or
validation). Reports real per-class Dice/IoU (Microaneurysm, Haemorrhage, Hard Exudate, Soft
Exudate) plus their mean, via `LesionSegmentationStage.evaluate()` (which reuses
`training.metrics.dice_coefficient`/`iou_score` per class, unmodified). No number here is
fabricated or estimated -- if training hasn't run, this cell reports that plainly instead.

In [ ]:
if history is None:
    print("Skipping evaluation -- training has not been run yet (RUN_TRAINING was False in Section 11).")
    test_metrics = None
else:
    test_metrics = stage.evaluate(test_ds)
    print("Held-out IDRiD Testing Set metrics (27 images, official split, never used in training "
          "or validation) -- per-class + mean Dice/IoU:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.4f}")

## 13. Export

Runs only if a trained model exists. Exports the best checkpoint to
`exported_models/LesionSegmentation/best_model.keras` on Drive (`.keras`, per
`SEGMENTATION_ARCHITECTURE.md` Sec 6). Training's own `checkpoints/`/`logs/` already live under
this run's `experiments/LesionSegmentation/<timestamp>/` (via `TrainingConfig(run_dir=experiment.root)`
inside `LesionSegmentationStage.train()`); this section only writes the held-out test-set
evaluation metrics into this same experiment's `evaluation/` folder -- the Colab-appropriate,
Drive-persistent "existing lesion-segmentation results location" for a Colab run (see
`PROJECT_STRUCTURE.md`'s Output Locations table: `results/<module>/` is the local/CLI-only
convention `config.LESION_SEG_RESULTS_DIR` resolves to; Colab runs use
`experiments/<Module>/<timestamp>/evaluation/` instead, consistent with the "Drive only, `/content`
is temporary staging" storage rule).

In [ ]:
import json

if history is None:
    print("Skipping export -- no trained model to export (training has not been run yet).")
    exported_path = None
else:
    exported_path = os.path.join(colab_config.DRIVE.exported_model_dir("LesionSegmentation"), "best_model.keras")
    stage.save(exported_path)
    print(f"Exported best model to {exported_path}")

    eval_results_path = os.path.join(experiment.evaluation_dir, "test_set_metrics.json")
    with open(eval_results_path, "w") as f:
        json.dump(test_metrics, f, indent=2)
    print(f"Saved held-out test-set evaluation metrics to {eval_results_path}")

## 14. Final Summary

Clearly distinguishes validation metrics (from training, on the 20% split carved out of the
54-image Training Set) from the final held-out test metrics (the official 27-image Testing Set,
Section 12) -- these are never the same numbers and must not be conflated.

In [ ]:
print("=" * 72)
print("Stage 04: Lesion Segmentation -- Summary")
print("=" * 72)
print(f"Model: Attention U-Net, input {(*IMAGE_SIZE, 4)}, 4 sigmoid output channels")
print(f"Lesion classes (output-channel order): {list(LESION_CLASSES)}")
print(f"Vessel checkpoint used: {DEFAULT_VESSEL_MODEL_PATH}")
print(f"Training configuration: epochs={EPOCHS} batch_size={BATCH_SIZE} "
      f"learning_rate={LEARNING_RATE} val_split={VAL_SPLIT} image_size={IMAGE_SIZE} seed={RANDOM_SEED}")
print(f"Experiment root: {experiment.root}")

if history is None:
    print("\nTRAINING WAS NOT RUN in this notebook session (RUN_TRAINING=False in Section 11).")
    print("No model was exported and no evaluation metrics exist yet.")
else:
    print(f"\nExported model: {exported_path}")

    val_metrics = {k: v[-1] for k, v in history.history.items() if k.startswith("val_")}
    print("\nVALIDATION metrics (last epoch, on the 20% split carved out of the 54-image "
          "Training Set -- NOT the official held-out test set):")
    for k, v in val_metrics.items():
        print(f"  {k}: {v:.4f}")

    print("\nFINAL HELD-OUT TEST-SET metrics (27-image official IDRiD segmentation Testing Set, "
          "never used in training or validation) -- per-class + mean Dice/IoU:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.4f}")

print("\nCaveats:")
print(f"- IDRiD's segmentation subset is small (54 training images, split {1 - VAL_SPLIT:.0%}/"
      f"{VAL_SPLIT:.0%} train/val) -- watch for overfitting between train and val metrics.")
print("- Soft Exudate ground truth is sparse (not every image has one) -- its Dice/IoU is "
      "expected to be noisier than the other three classes.")
print("- Stage 03's vessel channel is a pretrained, DRIVE-tuned model applied to Stage 02's "
      "Gamma+CLAHE output -- a documented distribution-shift caveat inherited from Stage 03 "
      "(SEGMENTATION_ARCHITECTURE.md Sec 1.2), unchanged and unresolved here.")
print("- No metric above is fabricated or estimated -- every number printed here comes directly "
      "from this session's actual trained model, or is explicitly marked as not yet available.")